# Traffic Signs Detection Training with YOLOv8

## Overview
This notebook trains a YOLOv8 model for traffic signs detection using Kaggle's GPU resources.

### Key Features:
- **Multi-GPU Training**: Utilizes 2x Tesla T4 GPUs for faster training
- **AdamW Optimizer**: Better convergence than SGD
- **Auto Dataset Detection**: Automatically detects YOLO format datasets
- **Optimized Parameters**: Learning rate, weight decay, and augmentation tuned for traffic signs

### Dataset
- **Source**: Traffic Signs Dataset in YOLO format from Kaggle
- **Classes**: 4 categories (speed limit, yield, mandatory, other)
- **Format**: YOLO bounding box annotations


In [ ]:
# Environment Setup + Hardware Check
import sys, torch
print("Python:", sys.version)
print("CUDA available:", torch.cuda.is_available())
print("GPU(s):", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

# Install required packages (avoid torch/torchvision conflicts)
# !pip -q install -U ultralytics opencv-python wandb


In [ ]:
# Kaggle API Configuration
import os, glob, json, shutil, pathlib

# Setup Kaggle API if kaggle.json is attached via "Add data"
kc = pathlib.Path("/root/.config/kaggle")
kc.mkdir(parents=True, exist_ok=True)

matches = glob.glob("/kaggle/input/**/kaggle.json", recursive=True)
if matches:
    shutil.copy(matches[0], kc/"kaggle.json")
    os.chmod(kc/"kaggle.json", 0o600)
    creds = json.load(open(kc/"kaggle.json"))
    os.environ["KAGGLE_USERNAME"] = creds.get("username", "")
    os.environ["KAGGLE_KEY"] = creds.get("key", "")
    print("✅ Kaggle API configured from:", matches[0])
else:
    print("ℹ️ No kaggle.json found in /kaggle/input — OK if dataset is already added as Input.")

In [ ]:
# Auto-detect YOLO Traffic Signs Dataset
import os, glob, pathlib, yaml, subprocess

def find_yolo_roots(base_dir: str):
    """Find YOLO dataset roots by looking for train/images or images/train patterns"""
    roots = set()
    for pat in ("**/train/images", "**/images/train"):
        for p in pathlib.Path(base_dir).glob(pat):
            roots.add(p.parents[1])  # dataset root
    return sorted(roots, key=lambda p: len(str(p)))  # shorter paths first

# 1) Search in /kaggle/input for YOLO-structured datasets
candidates = find_yolo_roots("/kaggle/input")

source = None
if candidates:
    DATA_ROOT = str(candidates[0])
    source = "input"
    print(f"✅ Found YOLO dataset in input: {DATA_ROOT}")
else:
    # 2) Download via Kaggle API if not found in input
    print("📥 No YOLO-structured dataset found in /kaggle/input → downloading via Kaggle API...")
    work = pathlib.Path("/kaggle/working")
    os.chdir(work)
    dst = work/"data/traffic_signs"
    dst.mkdir(parents=True, exist_ok=True)
    
    # Download traffic signs dataset
    r = subprocess.run([
        "kaggle", "datasets", "download", "-d", 
        "valentynsichkar/traffic-signs-dataset-in-yolo-format", "-p", "/kaggle/working"
    ], check=False)
    
    if r.returncode != 0:
        raise SystemExit("❌ No YOLO dataset in /kaggle/input and API download failed. Please add dataset: 'valentynsichkar/traffic-signs-dataset-in-yolo-format'.")
    
    # Extract dataset
    subprocess.run([
        "bash", "-lc", 
        "unzip -o /kaggle/working/traffic-signs-dataset-in-yolo-format.zip -d /kaggle/working/data/traffic_signs >/dev/null"
    ], check=False)
    
    pathlib.Path("/kaggle/working/traffic-signs-dataset-in-yolo-format.zip").unlink(missing_ok=True)
    
    candidates = find_yolo_roots("/kaggle/working/data/traffic_signs")
    assert candidates, "❌ Downloaded and extracted but still no train/images found."
    DATA_ROOT = str(candidates[0])
    source = "downloaded"
    print(f"✅ Downloaded and extracted to: {DATA_ROOT}")

# Detect dataset structure
def has(p): 
    return os.path.isdir(os.path.join(DATA_ROOT, p))

train = "train/images" if has("train/images") else "images/train"
val = "valid/images" if has("valid/images") else ("val/images" if has("val/images") else None)

if not val:
    # If no validation set, use test or create from train
    val = "test/images" if has("test/images") else train
    print("⚠️ No validation set found, using:", val)

test = "test/images" if has("test/images") else val

print(f"📁 Dataset structure: train={train}, val={val}, test={test}")


In [ ]:
# Create Traffic Signs Dataset Configuration
os.makedirs("configs", exist_ok=True)
DATA_YAML = "configs/traffic_signs_detection.yaml"

# Traffic signs classes (4 main categories)
traffic_signs_classes = [
    "speed_limit",
    "yield",
    "mandatory",
    "other"
]

# Create YAML configuration
config = {
    "path": DATA_ROOT,
    "train": train,
    "val": val,
    "test": test,
    "nc": len(traffic_signs_classes),
    "names": traffic_signs_classes
}

with open(DATA_YAML, "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print("✅ Created configuration file:", DATA_YAML)
print("📄 Configuration content:")
print(open(DATA_YAML).read())
print(f"📊 Source: {source} | Dataset Root: {DATA_ROOT}")

In [ ]:
# Install/Update Required Packages
# Ensure compatible versions for stable training
!pip install -q "numpy<2.1,>=1.26.4" "matplotlib>=3.7,<3.9" "protobuf>=4.25.1" --upgrade

In [ ]:
# Traffic Signs Detection Training
from ultralytics import YOLO
import torch, os

# Disable W&B to avoid manual input (change 'true' -> 'false' and add wandb.login() to enable)
os.environ['WANDB_DISABLED'] = 'true'

# GPU Configuration
ngpu = torch.cuda.device_count()
device = "0,1" if ngpu >= 2 else (0 if ngpu == 1 else "cpu")
batch = 16 * (2 if ngpu >= 2 else 1)  # Scale batch size with GPU count

print(f"🚀 Training Configuration:")
print(f"   GPUs: {ngpu} | Device: {device} | Batch Size: {batch}")
print(f"   Model: YOLOv8n | Task: Traffic Signs Detection")

# Load pre-trained YOLOv8n model
model = YOLO("yolov8n.pt")

# Start training with optimized parameters for traffic signs
print("\n🎯 Starting Traffic Signs Detection Training...")
results = model.train(
    # Dataset Configuration
    data="configs/traffic_signs_detection.yaml",
    
    # Training Parameters
    epochs=50,              # More epochs for traffic signs (smaller objects)
    batch=batch,            # Dynamic batch size based on GPU count
    imgsz=640,              # Standard YOLO input size
    device=device,          # Multi-GPU if available
    workers=4,              # Data loading workers
    
    # Output Configuration
    project="runs_detect",
    name="traffic_signs_detection",
    
    # Optimizer Settings (optimized for traffic signs)
    optimizer="AdamW",      # Better convergence than SGD
    lr0=0.001,              # Lower learning rate for stability
    lrf=0.01,               # Final learning rate factor
    weight_decay=0.0005,    # L2 regularization
    
    # Training Control
    patience=15,            # Early stopping patience
    plots=True,             # Generate training plots
    verbose=True,           # Detailed logging
    
    # Data Augmentation (tuned for traffic signs)
    hsv_h=0.01,             # Hue augmentation (less for signs)
    hsv_s=0.5,              # Saturation augmentation
    hsv_v=0.3,              # Value augmentation
    degrees=5.0,            # Rotation (small for signs)
    translate=0.05,         # Translation (small for signs)
    scale=0.3,              # Scale augmentation
    fliplr=0.5,             # Horizontal flip
    mosaic=0.8,             # Mosaic augmentation
    mixup=0.1               # Mixup augmentation
)

print("\n✅ Training completed successfully!")
print(f"📁 Results saved to: runs_detect/traffic_signs_detection")

In [ ]:
# Training Results Analysis
import matplotlib.pyplot as plt
from IPython.display import Image, display
import os

results_dir = "runs_detect/traffic_signs_detection"

print("📊 Training Results Analysis")
print("=" * 50)

# Display training curves
if os.path.exists(f"{results_dir}/results.png"):
    print("📈 Training Curves:")
    display(Image(f"{results_dir}/results.png"))
else:
    print("⚠️ Training curves not found")

# Display confusion matrix
if os.path.exists(f"{results_dir}/confusion_matrix.png"):
    print("\n🎯 Confusion Matrix:")
    display(Image(f"{results_dir}/confusion_matrix.png"))
else:
    print("⚠️ Confusion matrix not found")

# Display validation predictions
if os.path.exists(f"{results_dir}/val_batch0_pred.jpg"):
    print("\n🔍 Validation Predictions Sample:")
    display(Image(f"{results_dir}/val_batch0_pred.jpg"))
else:
    print("⚠️ Validation predictions not found")

# Print final metrics
if hasattr(results, 'results_dict'):
    print("\n📋 Final Training Metrics:")
    for key, value in results.results_dict.items():
        if isinstance(value, (int, float)):
            print(f"   {key}: {value:.4f}")

print("\n🎉 Training Analysis Complete!")

In [ ]:
# Model Validation and Testing
print("🧪 Model Validation and Testing")
print("=" * 40)

# Load the best trained model
best_model_path = f"{results_dir}/weights/best.pt"

if os.path.exists(best_model_path):
    print(f"📦 Loading best model: {best_model_path}")
    best_model = YOLO(best_model_path)
    
    # Validate on test set
    print("\n🔬 Running validation on test set...")
    val_results = best_model.val(data="configs/traffic_signs_detection.yaml")
    
    print("\n📊 Validation Results:")
    print(f"   mAP50: {val_results.box.map50:.4f}")
    print(f"   mAP50-95: {val_results.box.map:.4f}")
    print(f"   Precision: {val_results.box.mp:.4f}")
    print(f"   Recall: {val_results.box.mr:.4f}")
    
    # Export model for deployment
    print("\n📤 Exporting model for deployment...")
    best_model.export(format="onnx")  # Export to ONNX for deployment
    print("✅ Model exported to ONNX format")
    
else:
    print(f"❌ Best model not found at: {best_model_path}")

print("\n🎯 Validation and Testing Complete!")

In [ ]:
# Create Results Archive
import zipfile
import shutil

print("📦 Creating Results Archive")
print("=" * 30)

archive_name = "traffic_signs_detection_results.zip"

if os.path.exists(results_dir):
    print(f"🗜️ Compressing {results_dir} to {archive_name}...")
    
    with zipfile.ZipFile(archive_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(results_dir):
            for file in files:
                file_path = os.path.join(root, file)
                arc_path = os.path.relpath(file_path, os.path.dirname(results_dir))
                zipf.write(file_path, arc_path)
    
    print(f"✅ Results archived to: {archive_name}")
    print(f"📊 Archive size: {os.path.getsize(archive_name) / (1024*1024):.2f} MB")
    
    # List archive contents
    print("\n📋 Archive Contents:")
    with zipfile.ZipFile(archive_name, 'r') as zipf:
        for info in zipf.infolist()[:10]:  # Show first 10 files
            print(f"   {info.filename} ({info.file_size} bytes)")
        if len(zipf.infolist()) > 10:
            print(f"   ... and {len(zipf.infolist()) - 10} more files")

else:
    print(f"❌ Results directory not found: {results_dir}")

print("\n🎉 Archive Creation Complete!")
print("\n💡 Download the archive to get all training results, models, and visualizations.")